# 260509 promotion split

`Membership_v2.csv`의 `is_promotion` 값을 기준으로 데이터를 2개 집단으로 분리한다.

- `promotion_1`: `is_promotion == 1`인 멤버십 행에 등장한 `USER_KEY` 기준
- `promotion_0`: `is_promotion == 0`인 멤버십 행에 등장한 `USER_KEY` 기준
- `View_History_v2.csv`는 `User_Mapping_v2.csv`를 거쳐 `USER_NUM` 기준으로 필터링한다.
- `Movie_Master_v2.csv`는 각 집단의 시청 이력에 등장한 `MOVIE_NUM`만 남긴다.

In [7]:
from pathlib import Path

import pandas as pd


SOURCE_DIR = Path.cwd().parent / "260509_view_delete"
OUTPUT_DIR = Path.cwd()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

membership = pd.read_csv(SOURCE_DIR / "Membership_v2.csv")
user_mapping = pd.read_csv(SOURCE_DIR / "User_Mapping_v2.csv")
view_history = pd.read_csv(SOURCE_DIR / "View_History_v2.csv")
movie_master = pd.read_csv(SOURCE_DIR / "Movie_Master_v2.csv")

print("membership:   ", membership.shape)
print("user_mapping: ", user_mapping.shape)
print("view_history: ", view_history.shape)
print("movie_master: ", movie_master.shape)

membership:    (23343, 15)
user_mapping:  (23720, 2)
view_history:  (175301, 5)
movie_master:  (14502, 4)


In [8]:
print("is_promotion 분포:")
print(membership["is_promotion"].value_counts(dropna=False))

is_promotion 분포:
is_promotion
1    11955
0    11388
Name: count, dtype: int64


In [9]:
def split_by_promotion(is_promotion: int) -> dict[str, pd.DataFrame]:
    membership_split = membership.loc[
        membership["is_promotion"].eq(is_promotion)
    ].copy()

    user_keys = membership_split["USER_KEY"].dropna().unique()

    user_mapping_split = user_mapping.loc[
        user_mapping["USER_KEY"].isin(user_keys)
    ].copy()

    user_nums = user_mapping_split["USER_NUM"].dropna().unique()

    view_history_split = view_history.loc[
        view_history["USER_NUM"].isin(user_nums)
    ].copy()

    movie_nums = view_history_split["MOVIE_NUM"].dropna().unique()

    movie_master_split = movie_master.loc[
        movie_master["MOVIE_NUM"].isin(movie_nums)
    ].copy()

    return {
        "membership": membership_split,
        "user_mapping": user_mapping_split,
        "view_history": view_history_split,
        "movie_master": movie_master_split,
    }


promotion_1 = split_by_promotion(1)
promotion_0 = split_by_promotion(0)

summary = []
for group_name, group_data in {"promotion_1": promotion_1, "promotion_0": promotion_0}.items():
    for data_name, data in group_data.items():
        summary.append({"group": group_name, "data": data_name, "rows": len(data), "columns": len(data.columns)})

pd.DataFrame(summary)

,group,data,rows,columns
0,promotion_1,membership,11955,15
1,promotion_1,user_mapping,11957,2
2,promotion_1,view_history,88705,5
3,promotion_1,movie_master,4914,4
4,promotion_0,membership,11388,15
5,promotion_0,user_mapping,11261,2
6,promotion_0,view_history,82824,5
7,promotion_0,movie_master,4831,4


In [10]:
output_map = {
    "promotion_1_membership_v2.csv":   promotion_1["membership"],
    "promotion_1_user_mapping_v2.csv": promotion_1["user_mapping"],
    "promotion_1_view_history_v2.csv": promotion_1["view_history"],
    "promotion_1_movie_master_v2.csv": promotion_1["movie_master"],
    "promotion_0_membership_v2.csv":   promotion_0["membership"],
    "promotion_0_user_mapping_v2.csv": promotion_0["user_mapping"],
    "promotion_0_view_history_v2.csv": promotion_0["view_history"],
    "promotion_0_movie_master_v2.csv": promotion_0["movie_master"],
}

for file_name, data in output_map.items():
    output_path = OUTPUT_DIR / file_name
    data.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"saved: {output_path.name}  rows={len(data):,}")

saved: promotion_1_membership_v2.csv  rows=11,955
saved: promotion_1_user_mapping_v2.csv  rows=11,957
saved: promotion_1_view_history_v2.csv  rows=88,705
saved: promotion_1_movie_master_v2.csv  rows=4,914
saved: promotion_0_membership_v2.csv  rows=11,388
saved: promotion_0_user_mapping_v2.csv  rows=11,261
saved: promotion_0_view_history_v2.csv  rows=82,824
saved: promotion_0_movie_master_v2.csv  rows=4,831


In [11]:
promotion_1_user_keys = set(promotion_1["membership"]["USER_KEY"])
promotion_0_user_keys = set(promotion_0["membership"]["USER_KEY"])
overlap_user_keys = promotion_1_user_keys & promotion_0_user_keys

print("promotion_1 USER_KEY 수:", len(promotion_1_user_keys))
print("promotion_0 USER_KEY 수:", len(promotion_0_user_keys))
print("두 집단에 모두 등장하는 USER_KEY 수:", len(overlap_user_keys))

promotion_1 USER_KEY 수: 11951
promotion_0 USER_KEY 수: 11221
두 집단에 모두 등장하는 USER_KEY 수: 38


In [12]:
# 겹치는 USER_KEY의 Membership 행 확인
overlap_rows = membership.loc[
    membership["USER_KEY"].isin(overlap_user_keys)
].sort_values(["USER_KEY", "is_promotion"])

print(f"겹치는 USER_KEY {len(overlap_user_keys)}명의 Membership 행 수: {len(overlap_rows)}")
print("→ 동일 USER_KEY가 is_promotion=1 행과 is_promotion=0 행을 모두 가짐\n")
overlap_rows[["USER_KEY", "is_promotion", "reg_date", "end_date"]]

겹치는 USER_KEY 38명의 Membership 행 수: 85
→ 동일 USER_KEY가 is_promotion=1 행과 is_promotion=0 행을 모두 가짐



,USER_KEY,is_promotion,reg_date,end_date
11165,01b73db46b0b3f6c24ca22dc4629ed41ed0d6ac4beccf8...,0,2014-03-21,2014-04-21
10666,01b73db46b0b3f6c24ca22dc4629ed41ed0d6ac4beccf8...,1,2014-03-21,2014-03-21
11676,06e29aff8528056461562d5763bded5505f51c9b893148...,0,2006-03-21,2006-04-21
341,06e29aff8528056461562d5763bded5505f51c9b893148...,1,2006-03-21,2006-03-21
497,07e21f3a885e79ef644b536bbc1394a08874766e979f79...,0,2003-03-21,2003-04-21
...,...,...,...,...
1949,eb54ac763a0738f16cc1e8ace7d33621860059b136c34c...,1,2005-03-21,2006-03-21
6645,eed0de36a9728a8b87dd1060ca2decf19a55276c5dbe8f...,0,2012-03-21,2013-04-21
14784,eed0de36a9728a8b87dd1060ca2decf19a55276c5dbe8f...,1,2008-03-21,2008-04-21
12226,f06951ff35605cfa7796e238a35482c0593a7dc566ef49...,0,2014-03-21,2014-04-21
